# AMSA Vessel Tracking: Comprehensive Strategic Analysis (2012–2025)

|  |  |
| ----------- | ----------- |
| Author:| Karishma Khanna |
| Edited:| 2026-01-08 |
| Affiliation:| AODN/IMOS |
| e-mail:| karishma.khanna@utas.edu.au |
| Date of creation:| 2026-01-08 |
| Date of last update:| 2026-01-08 |



## Background

This is a template notebook outlining the library stack and content structure for NESP Data Products.

It outlines:
1. [Connecting to NESP 5.9 datasets](#connecting-to-nesp-59-datasets)
2. [H3 Aggregation and Mapping](#h3-aggregation-and-mapping)
3. [Identifying Trend Sites](#identifying-trend-sites)

## NESP 5.9 Datasets

| Dataset name | Description | Metadata | S3 URL |
| ------------ | ----------- | -------- | ------ |
| AMSA         | AMSA Vessel Tracking               | -     | [link](s3://data-uplift-public/stored/datauplift/amsa/)                                   |
| Kelp         | Squidle+ Kelp Annotations          | -     | [link](s3://data-uplift-public/stored/datauplift/kelp/kelp.parquet)                       |
| NRMN         | NRMN Reef Life Surveys             | -     | [link](s3://data-uplift-public/stored/datauplift/nrmn/nrmn.parquet)    |
| Seabird      | Seabird Observations and Tracking  | -     | [link](s3://data-uplift-public/stored/datauplift/seabird/seabird.parquet)                 |
| Seagrass     | Seagrass Surveys                   | -     | [link](s3://data-uplift-public/stored/datauplift/seagrass/seagrass.parquet)               |

### S3 Details
NESP 5.9 datasets are currently stored in S3 and are publicly available as per the following table:

| Bucket               | Key                                                      | Partitioned |
| -------------------- | -------------------------------------------------------- | ----------- |
| `data-uplift-public` | `stored/datauplift/amsa/year=*/source=*/*.parquet`       | [x]         |
| `data-uplift-public` | `stored/datauplift/kelp/kelp.parquet`                    | [ ]         |
| `data-uplift-public` | `stored/datauplift/nrmn/nrmn.parquet` | [ ]         |
| `data-uplift-public` | `stored/datauplift/seabird/seabird.parquet`              | [ ]         |
| `data-uplift-public` | `stored/datauplift/seagrass/seagrass.parquet`            | [ ]         |

## Required Python & Packages
The required packages are found in the [pyproject.toml](pyproject.toml) under `dependencies`.

### `uv` installation
It is highly recommended to manage python version, environment and projects with [uv](https://github.com/astral-sh/uv).

```bash
uv venv && source .venv/bin/activate && uv pip install .
```

### `pip` installation
Otherwise, native python tooling can be used to manage the python environment and project. 

*Note that python3.12 or greater is required.*

```bash
python -m venv .venv && source .venv/bin/activate && pip install .
```

### Running Notebooks in Jupyter or IDE
The notebook can be manually run by launching jupyter notebook from the console:
```bash
jupyter notebook
```
Otherwise, if running in an IDE, select the newly created `.venv` as the kernal.


## Notebook Stack

### IO
`pyarrow` is the defacto parquet and arrow standard library tooling in the python eco-system, offering exceptional performance, documentation and stability.

Its `fs` class allows connecting to parquet datasets in AWS S3, which is perfect for our use case.

### Computation
`polars` is fast replacing pandas as the primary python dataframe API. It is more tightly integrated with arrow (and pyarrow), has exceptional performance, greater than memory dataset manipulation, a more consistent and declarative API, built in plotting and many other advantages.

### Mapping
`pydeck` is a python binding for `deck.gl`, a GPU-powered framework for visual exploratory data analysis of large datasets. It is optimised for Jupyter notebooks and highly customisable.

Standard mapping tools like `folium` struggle to handle the number of mapped shapes, so we favor the more optimised `pydeck` library.

### Plotting
`seaborn` and `matplotlib` are used for statistical data visualization. `seaborn` provides a high-level interface for drawing attractive and informative statistical graphics, while `matplotlib` manages the figure layouts and axis formatting.

### Rich
TODO: Update on `rich` library

In [ ]:
# Set up rich console
import rich
import rich.console
console = rich.console.Console()

## Connecting to NESP 5.9 Datasets

NESP 5.9 datasets can be connected to via a pyarrow `S3FileSystem`.

The `S3FileSystem` manages the connection to the NESP 5.9 datasets stored in S3, resulting in a `pyarrow.Dataset` handle that we may use for efficient loading and subsetting of the dataset.

Note that a dataset can be comprised of a single parquet file or partitioned (into separate parquet fragments/files).

This step also makes the dataset schema available and file structure available allowing for pre-filtering without downloading the entire dataset.

Finally, we may estimate the in memory dataset size by sampling the dataset. If the dataset is small, we may use a `DataFrame` straight away. Otherwise if the dataset is too large, we must pre-aggregate and or filter using a `LazyFrame` before loading the dataset into memory.

In [ ]:
import pyarrow
import pyarrow.fs
import pyarrow.dataset
from nesp import util

# --- FileSystem Configuration ---
# Initialize S3FileSystem for ap-southeast-2 region.
# Set anonymous=True for public bucket access.

FILE_SYSTEM = pyarrow.fs.S3FileSystem(
    region="ap-southeast-2", 
    anonymous=True,
)

# --- Dataset Connection ---
# Create a PyArrow Dataset handle for the AMSA source.
# This performs a lazy connection; data is not loaded into memory yet.
ds = pyarrow.dataset.dataset(
    source="data-uplift-public/stored/datauplift/amsa/dataset",
    filesystem=FILE_SYSTEM,
)

# --- Schema Inspection ---

console.log("📂 Inspecting AMSA Archive...")
rich.print('Total Inspected files: ',len(list(ds.get_fragments())))

# Display schema definition and units.
schema_rich_table = util.print_schema_rich_table(
    schema=ds.schema,
    metadata_keys=[
        "definition",
        "unit"
    ],
)

console.print(schema_rich_table)

## Loading Data
The dataset handle is our connection to data in S3. We can use this handle to load data directly into a `DataFrame` or `LazyFrame`

### DataFrame
For smaller datasets the `DataFrame` is preferrable. This loads the entire dataset into memory ready for analysis.

### LazyFrame
For larger datasets the `DataFrame` may not fit into memory. In these cases the `LazyFrame` is ideal as it allows the dataset to be filtered and or aggregated prior to loading the dataset, allowing it to fit into memory, ready for analysis.

> **For AMSA:** Given the dataset exceeds available RAM, we initialize a `LazyFrame`.

In [ ]:
import polars as pl
import polars_h3
import polars 
import os

# --- Initialize LazyFrame ---
# scan_pyarrow_dataset wraps the existing dataset handle.
# No data is transferred until action (collect) is called.
amsa_lf = pl.scan_pyarrow_dataset(ds)

print("✅ LazyFrame initialized.")

## H3 Aggregation and Mapping

The NESP 5.9 datasets are aggregated data products that cover large regions and time periods. It is useful to first aggregate the datasets so that we may understand their temporal-spatial bounds.

### H3 Indexing
The `h3` library allows us to spatially index datasets at different [resolutions](https://h3geo.org/docs/core-library/restable#average-area-in-km2) (zoom levels) according to the record latitude and longitude.

Additionally, the `polars_h3` library offers a polars style interface for faster tagging with `polars.DataFrames`.

### H3 Mapping
The `pydeck` library offers a built in [`H3HexagonLayer`](https://deck.gl/docs/api-reference/geo-layers/h3-hexagon-layer) layer type that greatly reduces complexity in mapping H3 Shapes.

> **Visualization Strategy:**
> * **Log-Scaling:** Maritime traffic density varies wildly (Power Law). We apply a $Log_{10}$ transform to make both ports and open oceans visible.
> * **Global Max:** We calculate the maximum density across *all years* to create a fixed color scale. This ensures that "Red" in 2013 represents the same density as "Red" in 2024, allowing for valid comparisons.

In [ ]:
console.log("🚀 Starting H3 Aggregation...")

# --- H3 Aggregation ---
# Group by the pre-calculated Resolution 8 Index and Year.
h3_trend_df = (
    amsa_lf.group_by(
        polars.col("h3Index").alias("h3IndexResolution8"),
        polars.col("eventDate").dt.year().alias("year"),
    )
    .agg(
        polars.len().alias("n_records"),
    )
    # The collect statement applies the above aggregation and streams
    # only the required data to the client (us)
    .collect(engine="streaming")
)

console.log(f"✅ Aggregation Complete.")

## Visualising the H3 Aggregation

With the data aggregated and the global scale determined, we can now render the spatial density map using `pydeck`.

The `generate_comparative_map` function below handles:
1.  **Filtering:** Selecting the specific year from the aggregated dataset.
2.  **Normalization:** Mapping the Log-Score to a fixed color palette using the `GLOBAL_MAX_LOG` calculated previously.
3.  **Rendering:** Generating the 3D Hexagon Layer.

In [ ]:
import pydeck

def generate_comparative_map(dataframe, target_year):
    """
    Generates a pydeck H3 Hexagon map for a specific year using a fixed global color scale.

    This function filters a Polars DataFrame for a specific year, bins the data into 
    discrete color buckets based on a global maximum value to ensure visual consistency 
    across different time slices, and renders the result using pydeck.

    :param dataframe: The input data containing H3 indices and record counts.
    :type dataframe: polars.LazyFrame or polars.DataFrame
    :param target_year: The specific year to filter and visualize.
    :type target_year: int
    :return: A pydeck Deck object rendered in the current environment.
    :rtype: pydeck.bindings.deck.Deck
    """
    
    console.log(f"🎨 Generating Map for {target_year}...")

    # --- Filter Data ---
    # Filter the LazyFrame for the target year.
    year_df = dataframe.filter(
        polars.col("year").eq(target_year),
    )

    # # --- Color Palette ---
    # # Generate a 10-step discrete color palette ('plasma') for high contrast.
    n_quantiles = 10

    # --- Layer Construction ---
    # Create a separate H3HexagonLayer for each color bucket.
    # This is required because pydeck layers typically take a single fill color.
    layers = util.generate_pydeck_hexagon_layers(
        df=year_df,
        aggregate_column_name="n_records",
        hexagon_index_column_name="h3IndexResolution8",
        n_quantiles=n_quantiles,
        color_palette="plasma",
    )

    # --- View State & Deck ---
    # Dynamically center view as per data
    view_state = util.generate_view_state(
        df=year_df.with_columns(
            polars_h3.cell_to_lat(polars.col("h3IndexResolution8")).alias("decimalLatitude"),
            polars_h3.cell_to_lng(polars.col("h3IndexResolution8")).alias("decimalLongitude"),
        ),
    )
    
    r = pydeck.Deck(
        layers=layers,
        initial_view_state=view_state,
        map_style=pydeck.map_styles.CARTO_LIGHT, # White background
        tooltip={
            "html": "<b>H3:</b> {h3IndexResolution8}<br/><b>Pings:</b> {n_records}<br/>"
        }
    )
    
    return r.show()


In [ ]:
# The map for year 2013
print("--- 2013 Vessel Density ---")
generate_comparative_map(h3_trend_df, 2013)

In [ ]:
# The map for year 2019
print("--- 2019 Vessel Density ---")
generate_comparative_map(h3_trend_df, 2019)

In [ ]:
# The map for year 2024
print("--- 2024 Vessel Density ---")
generate_comparative_map(h3_trend_df, 2024)

## Regional Analysis: Sydney Harbour

While the global map uses a coarse resolution (Resolution 8, approx 0.7km²) suitable for open ocean shipping lanes, complex waterways like Sydney Harbour require finer detail.

To analyze this region effectively, we perform a **Spatial Deep Dive**:
1.  **Spatial Filter:** We restrict the query to the Sydney bounding box.
2.  **Re-Indexing:** We calculate a new H3 Index at **Resolution 9** (approx 0.1km²). This allows us to resolve individual ferry routes and harbour traffic that would otherwise be merged into a single block.

In [ ]:
# To minimise memory footprint, load a where and when df
temporal_spatial_df = amsa_lf.select(
    polars.col("eventDate"),
    polars.col("decimalLatitude"),
    polars.col("decimalLongitude"),
).collect()
temporal_spatial_df.estimated_size(unit="gb")
temporal_spatial_df.describe()

In [ ]:
def generate_high_res_map(
    df: polars.DataFrame,
    target_year: int,
    h3_resolution: int = 9,
    max_lat: float = -33.78,
    min_lat: float = -33.93,
    max_lon: float = 151.35,
    min_lon: float = 151.10,
):
    """
    Generate a high resolution h3 map.mro

    Defaults set up for Sydney.
    """
    
    console.log(f"⚓ Generating High-Res Map (Res {h3_resolution}) for {target_year}...")

    # Error check the bounds
    if max_lat <= min_lat:
        raise ValueError("max_lat must be greater than min_lat: found `{max_lat}` <= `{min_lat}`")
    if max_lon <= min_lon:
        raise ValueError("max_lon must be greater than min_lat: found `{max_lon}` <= `{min_lon}`")

    # --- Filter & Re-Index (Resolution 9) ---
    # We filter the main LazyFrame ('amsa_lf') to the specific region and year.
    # We calculate a new 'h3Index' at the specified new h3 resolution.
    map_df = (
        df
        .filter(
            polars.col("decimalLatitude").is_between(min_lat, max_lat),
            polars.col("decimalLongitude").is_between(min_lon, max_lon),
            polars.col("eventDate").dt.year().eq(target_year),
        )
        .select(["decimalLatitude", "decimalLongitude"])
        .with_columns(
            polars_h3.latlng_to_cell(
                lat=polars.col("decimalLatitude"), 
                lng=polars.col("decimalLongitude"), 
                resolution=h3_resolution,
                return_dtype=polars.String
            ).alias("h3Index")
        )
        .group_by("h3Index")
        .agg(polars.len().alias("n_records"))
    )
    
    # --- Validation ---
    if map_df.height == 0:
        console.log("⚠️ No data found for this year/region.")
        return

    n_quantiles = 10
    layers = util.generate_pydeck_hexagon_layers(
        df=map_df,
        aggregate_column_name="n_records",
        hexagon_index_column_name="h3Index",
        n_quantiles=n_quantiles,
        color_palette="plasma",
    )

    # --- View State & Deck ---
    # Dynamically center view as per data
    view_state = util.generate_view_state(
        df=map_df.with_columns(
            polars_h3.cell_to_lat(polars.col("h3Index")).alias("decimalLatitude"),
            polars_h3.cell_to_lng(polars.col("h3Index")).alias("decimalLongitude"),
        ),
    )
    
    r = pydeck.Deck(
        layers=layers,
        initial_view_state=view_state,
        map_style=pydeck.map_styles.CARTO_LIGHT,
        tooltip={"text": "n_records: {n_records}"}
    )
    
    return r.show()

In [ ]:
# The map for year 2016
print("--- 2016 Vessel Density Map of Sydney Harbour---")
generate_high_res_map(temporal_spatial_df, 2016, h3_resolution=12)

In [ ]:
# The map for year 2024
print("--- 2016 Vessel Density Map of Sydney Harbour---")
generate_high_res_map(temporal_spatial_df, 2024, h3_resolution=12)

## Longitudinal Data Health

This section characterizes the dataset's **temporal stability**.

Long-term maritime datasets are frequently influenced by **"Structural Breaks"**—step-changes in data volume caused by upgrades to surveillance infrastructure (e.g., new satellite launches). The following analysis distinguishes between **actual fleet growth** (increased traffic) and **systematic growth** (improved sensor sensitivity).

### Methodology
To quantify these trends, we aggregate the data annually and derive three core metrics:
1.  **Volume:** The total count of AIS transmission records (Data Size).
2.  **Fleet:** The count of unique vessels identified by `craftID` (Traffic Size).
3.  **Intensity:** The average number of records per vessel (Sensor Resolution).

> **Normalization Strategy:** To compare trends across different scales (thousands of ships vs. millions of records), all metrics are indexed to their **2012 baseline (2012 = 100)**. This allows for a direct correlation analysis of relative growth rates.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# --- Aggregation ---
# We reuse the 'amsa_lf' LazyFrame initialized above and group by Year to extract high-level statistics without loading the raw data.

console.log("🚀 Calculating Annual Statistics (Streaming Mode)...")

yearly_stats = (
    amsa_lf.select(["eventDate", "craftID"])
    .with_columns(pl.col("eventDate").dt.year().alias("Year"))
    .group_by("Year")
    .agg([
        pl.col("craftID").n_unique().alias("Unique_Vessels"),      # Fleet Size
        pl.len().alias("Total_Records"),                           # Data Volume
        pl.col("eventDate").n_unique().alias("Unique_Timestamps")  # Temporal Granularity
    ])
    .sort("Year")
    .collect(streaming=True)    # Trigger computation
)

# --- Normalization (Base 100) ---
# To compare "Vessels" (thousands) vs "Records" (millions), we index them to their 2012 baseline. (2012 value = 100).

base_vessels = yearly_stats["Unique_Vessels"][0]
base_records = yearly_stats["Total_Records"][0]
base_timestamps = yearly_stats["Unique_Timestamps"][0]

yearly_stats = yearly_stats.with_columns([
    (pl.col("Total_Records") / pl.col("Unique_Vessels")).alias("Records_Per_Vessel"),
    (pl.col("Unique_Vessels") / base_vessels * 100).alias("Idx_Vessels"),
    (pl.col("Total_Records") / base_records * 100).alias("Idx_Records"),
    (pl.col("Unique_Timestamps") / base_timestamps * 100).alias("Idx_Timestamps"),
    # Calculate Ping Gap (Inverse of Frequency) for annotation
    (365 * 24 * 60 / pl.col("Unique_Timestamps")).alias("Avg_Gap_Minutes")
])

# --- Visualization ---
# We use Matplotlib to create a 2x2 Grid, and Seaborn to paint the data.

sns.set_theme(style="whitegrid")
df_plot = yearly_stats.to_pandas()  # Convert small agg stats to Pandas for plotting

# Function to format Y-axis as Millions
def millions(x, pos):
    return '%1.1fM' % (x * 1e-6)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.subplots_adjust(hspace=0.3)

# A. Volume (Bar Chart)
sns.barplot(data=df_plot, x="Year", y="Total_Records", ax=axes[0,0], color="#3498db", alpha=0.7)
axes[0,0].yaxis.set_major_formatter(FuncFormatter(millions))
axes[0,0].set_title("A. Total Data Volume (Records)", fontweight='bold')
axes[0,0].grid(axis='y', alpha=0.3)

# B. Fleet Size (Line Chart)
sns.lineplot(data=df_plot, x="Year", y="Unique_Vessels", marker="o", ax=axes[0,1], color="#e74c3c", linewidth=3)
axes[0,1].set_title("B. Unique Vessels", fontweight='bold')
axes[0,1].grid(True, alpha=0.3)

# C. Growth Correlation (Multi-Line)
# This graph reveals if data grew because *ships* increased or *sensors* improved.
ax_c = axes[1,0]
sns.lineplot(data=df_plot, x="Year", y="Idx_Vessels", ax=ax_c, color="#e74c3c", label="Vessel Growth", linewidth=2.5, marker="o")
sns.lineplot(data=df_plot, x="Year", y="Idx_Records", ax=ax_c, color="#3498db", label="Volume Growth (Records)", linewidth=2.5, linestyle="--")
sns.lineplot(data=df_plot, x="Year", y="Idx_Timestamps", ax=ax_c, color="#2ecc71", label="Frequency Growth (Timestamps)", linewidth=2.5, linestyle="-.")

ax_c.set_ylabel("Growth Index (Base = 100)")
ax_c.set_title("C. Correlation: Vessels vs. Volume vs. Frequency", fontweight='bold')
ax_c.legend(loc='upper left')

# Add "Ping Gap" Annotation
if len(df_plot) > 0:
    gap_start = df_plot["Avg_Gap_Minutes"].iloc[0]
    gap_end = df_plot["Avg_Gap_Minutes"].iloc[-1]
    last_year = df_plot["Year"].iloc[-1]
    last_val = df_plot["Idx_Timestamps"].iloc[-1]
    
    ax_c.annotate(f"Ping Gap improved:\n{gap_start:.1f}m -> {gap_end:.1f}m", 
                 xy=(last_year, last_val), 
                 xytext=(last_year - 4, last_val + 50), 
                 arrowprops=dict(facecolor='black', shrink=0.05),
                 fontsize=10, bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9))

# D. Intensity (Bar Chart)
sns.barplot(data=df_plot, x="Year", y="Records_Per_Vessel", ax=axes[1,1], color="#9b59b6", alpha=0.6)
axes[1,1].set_title("D. Tracking Intensity (Avg Records per Vessel)", fontweight='bold')
axes[1,1].grid(axis='y', alpha=0.3)

plt.suptitle("Longitudinal Data Growth & System Resolution Analysis", fontsize=18, y=0.95)
plt.show()

In [ ]:
# Check if the "Magic Column" is in your dataset
if "australianMarineRegionsTags" in amsa_lf.columns:
    print("✅ The column is here! It was inherited from the source.")
else:
    print("❌ The column is missing. We are looking at the wrong file.")

# Australian Marine Regions

Each occurrence in the dataset is tagged with its corresponding **Australian Marine Region**.
These tags allow us to filter and analyze traffic within specific legal boundaries (e.g., Marine Parks, Heritage Areas).

> **Note:** The tagging is accurate to ~6km at region boundaries.

### Interactive Explorer
Use the map below to explore the official boundaries of Commonwealth Marine Regions.

In [ ]:
import IPython.display

# Display the Commonwealth Marine Regions
# This allows us to visually explore the boundaries before analyzing them.
IPython.display.IFrame(
    src="https://fed.dcceew.gov.au/datasets/commonwealth-marine-regions/explore?location=-33.665700%2C140.517300%2C4",
    width="100%", 
    height="1000px",
)

## 7Exploring Available Marine Regions Tags

The `australianMarineRegionsTags` column contains pipe-delimited values (e.g., `Commonwealth Marine Region: South-east|Marine Park: Freycinet`).
To understand the traffic distribution, we split these tags and count the occurrences of each unique region.

In [ ]:
import polars as pl

marine_lf = (
    amsa_lf
    .filter(
        pl.col("australianMarineRegionsTags").str.contains(pattern="Commonwealth Marine Region:South-east", literal=True)
    )
    .group_by(
        pl.col("h3Index"),
        pl.col("eventDate").dt.year().alias("Year")
             )
    .agg(
        pl.len().alias("n_records")
    )
    .collect()
)


In [ ]:
print(f"✅ Data Collected. Rows: {len(marine_lf)}")

In [ ]:
# --- 2. Template Prep (Fixed) ---
map_plot = (
    marine_lf
    .group_by("h3Index")
    .agg([
        pl.sum("n_records").alias("n_records"),
        pl.lit("South-east Region").alias("datasets")
    ])
    .with_columns(
        polars_h3.cell_to_latlng("h3Index").alias("coords")
    )
    # FIX: Extract Lat/Lon from the List manually
    .with_columns(
        pl.col("coords").list.get(0).alias("decimalLatitude"),
        pl.col("coords").list.get(1).alias("decimalLongitude")
    )
    .drop("coords") # Clean up
)

# --- 3. Visualization ---
h3_layers = util.generate_pydeck_hexagon_layers(
    df=map_plot,
    aggregate_column_name="n_records",
    n_quantiles=10,
    color_palette="plasma",
)

In [ ]:
deck = pydeck.Deck(
    layers=h3_layers,
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    tooltip=util.TOOLTIP,
    initial_view_state=util.generate_view_state(
        df=map_plot,
        view_proportion=0.9
    )
)

deck.show()